#Fase 1: Estrutura Básica e Otimização de Desempenho

Passo 1: Crie um arquivo chamado meu_dashboard.py. Importe as bibliotecas necessárias (streamlit e pandas) e crie um título principal para a página utilizando a função st.title('Dashboard de Vendas').

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
import pandas as pd

st.title('Dashboard de Vendas')

2026-09-28 01:42:08.241 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-28 01:42:08.246 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-28 01:42:08.249 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

Passo 2: Crie uma função chamada carregar_dados() que lê um arquivo CSV (ex: vendas.csv) utilizando o Pandas.

Passo 3: Para evitar que o sistema leia o arquivo repetidamente a cada clique na tela, decore essa função adicionando @st.cache_data na linha imediatamente anterior a ela. Isso faz com que o resultado seja guardado na memória e a função só recalcule quando os argumentos mudarem, acelerando o aplicativo.

In [ ]:
@st.cache_data
def carregar_dados():
    df = pd.read_csv("vendas.csv", parse_dates=["data"])
    return df

2026-09-28 01:42:09.936 No runtime found, using MemoryCacheStorageManager


In [ ]:
df = carregar_dados()

#Fase 2: Layout e Filtros Laterais (Interatividade)

Passo 1: Crie um painel lateral utilizando o comando st.sidebar.title('Filtros').

In [ ]:
st.sidebar.title("Filtros")

Passo 2: Adicione um widget interativo de seleção múltipla para as categorias de produtos. Utilize st.sidebar.multiselect('Selecione as Categorias', options=lista_de_categorias).

In [ ]:
lista_categorias = df["categoria"].unique().tolist()
categorias_sel = st.sidebar.multiselect(
    "Selecione as Categorias",
    options=lista_categorias,
    default=lista_categorias,
)

Passo 3: Programe a regra de ouro do Streamlit: utilize o valor retornado pelo widget que você acabou de criar para filtrar o DataFrame do Pandas. Assim, quando o usuário alterar o filtro, a tela toda se atualizará com os dados recortados.

In [ ]:
df_filtrado = df[df["categoria"].isin(categorias_sel)]
st.write("Após filtro:", df_filtrado.shape[0], "linhas")

#Fase 3: Métricas em Destaque e Visualização de Dados

Passo 1: Divida a parte superior da tela principal em colunas proporcionais utilizando o comando col1, col2 = st.columns([1, 1]).

In [ ]:
col1, col2 = st.columns([1, 1])

Passo 2: Dentro do bloco with col1:, adicione um indicador numérico em destaque utilizando st.metric(label='Receita Total', valor=receita_calculada). Faça o mesmo para a col2 exibindo o Total de Pedidos.

In [ ]:
with col1:
    receita_total = df_filtrado["receita"].sum()
    st.metric(label="Receita Total", value=f"R$ {receita_total:,.2f}")

with col2:
    total_pedidos = len(df_filtrado)
    st.metric(label="Total de Pedidos", value=f"{total_pedidos:,}")

Passo 3: Crie uma área de navegação por abas utilizando aba1, aba2 = st.tabs(['Evolução Mensal', 'Tabela de Dados']).

In [ ]:
aba1, aba2 = st.tabs(["Evolução Mensal", "Tabela de Dados"])

Passo 4: Na aba1, agrupe os dados de receita por mês no Pandas e gere um gráfico nativo de área utilizando st.area_chart(dados_agrupados). Lembre-se que é crucial agrupar os dados antes de plotar para garantir um gráfico legível.

In [ ]:
with aba1:
    receita_mensal = (
        df_filtrado.set_index("data")["receita"]
        .resample("ME")
        .sum()
        .rename("receita")
    )
    st.area_chart(receita_mensal)

Passo 5: Na aba2, exiba o DataFrame original filtrado na forma de uma tabela interativa (com rolagem e ordenação) utilizando st.dataframe(df_filtrado). Adicione também um botão para exportação utilizando st.download_button() para permitir o download desse recorte em CSV.

In [ ]:
with aba2:
    st.dataframe(df_filtrado, use_container_width=True)

    csv_bytes = df_filtrado.to_csv(index=False).encode("utf-8")
    st.download_button(
        label="Baixar recorte em CSV",
        data=csv_bytes,
        file_name="recorte_vendas.csv",
        mime="text/csv",
    )

#Fase 4: Publicação na Nuvem (Deploy)

https://dash-interativo.streamlit.app/
